In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Load the Excel file
file_path = r"C:\Users\Miranda C?rdova\Downloads\Verano Cientifico\Datos_para trabajar\15_sib_rangos.xlsx" 
xls = pd.ExcelFile(file_path)

In [ ]:
# Ver hojas disponibles
print(xls.sheet_names)


In [ ]:
# Leer una hoja espec?fica como DataFrame
df = xls.parse('Sheet1')  # o el nombre correcto de la hoja


In [ ]:

# Crear una columna auxiliar con los pares ordenados (conjuntos)
df["sample_pair"] = df.apply(lambda row: frozenset([row["FAM_Sample_info"], row["VIC_Sample_info"]]), axis=1)
df1 =df
# Eliminar duplicados de esos pares (sin importar el orden)
df = df[~df.duplicated(subset="sample_pair")]

# Eliminar tambi?n donde ambos valores son exactamente iguales
df = df[df["FAM_Sample_info"] != df["VIC_Sample_info"]]

# Eliminar la columna auxiliar si ya no se necesita
df = df.drop(columns="sample_pair")

print("Filas luego del filtrado:", df.shape[0])



In [ ]:
df1.shape[0]

In [ ]:
df.columns

In [ ]:
df['True_positive'].unique()
#array(['BlindSeach', 'BlindSearch', 'ScreenMatch'], dtype=object)
#array(['Simulados', 'Real'], dtype=object)


In [ ]:
df2 =df

In [ ]:
df["Datos"].unique()

In [ ]:
df=df2

In [ ]:
df = df[
    (df['Datos'] == 'Simulados') 
]

In [ ]:
# Aseg?rate de que True_positive sea booleana
df["True_positive"] = df["True_positive"].astype(str).str.upper().map({"TRUE": True, "FALSE": False})

# Eliminar columnas no deseadas
df = df.drop(columns=["FAM_Sample_info", "VIC_Sample_info",'Hoja','Sheet',"File",'Marcadores','Software','Kinship'], errors='ignore')

# Seleccionar solo columnas num?ricas
num_cols = df.select_dtypes(include='number').columns.tolist()

# Si hay una columna de clasificaci?n (e.g., "Classification"), incl?yela
if "Classification" in df.columns:
    hue_var = "Classification"
else:
    hue_var = "True_positive"



In [ ]:
import numpy as np

# Transformar LR (Value) a log10, cuidando ceros o valores negativos
df["log10_Value"] = np.log10(df["Value"].replace(0, np.nan)).replace(-np.inf, 0).fillna(0)


In [ ]:
from sklearn.preprocessing import OrdinalEncoder
# Codificaci?n Ordinal para la variable de rangos
# Definir el orden correcto de las categor?as
range_categories = ['0-99', '100-999', '1000-9999', '10000-99999', '100000-999999', '1000000+']

ordinal_encoder = OrdinalEncoder(categories=[range_categories]) # Pasar categories en una lista
df['variable_rangos_encoded'] = ordinal_encoder.fit_transform(df[['Value_range']])


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
# Selecci?n de variables predictoras
#features = ["k1", "k2", "Kinship_codificada", "Share_allele", "log10_Value"]
features = ["k1", "k2", "variable_rangos_encoded", "Kinship_codificada","Share_allele"]

X = df[features]
y = df["True_positive"].astype(bool)

In [ ]:

# Divisi?n
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

# Modelo
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Resultados
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
features = ["k1", "k2", "Share_allele", "log10_Value"]
X = df[features]
y = df["True_positive"].astype(bool)

In [ ]:

# Divisi?n
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

# Modelo
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Resultados
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ? 1. Define las features y target
#features = ["k1", "k2", "Share_allele", "log10_Value"]
features = ["k1", "k2", "variable_rangos_encoded"]

target = "True_positive"  # Ajusta si tu columna tiene otro nombre


In [ ]:
# ? 2. Divide en entrenamiento y prueba
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ? 3. Crea y ajusta el modelo
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)

# ? 4. Predicciones
y_pred = rf.predict(X_test)

# ? 5. Reporte de m?tricas
print(classification_report(y_test, y_pred))

In [ ]:
# ? 6. Matriz de confusi?n
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["False", "True"], yticklabels=["False", "True"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Matriz de Confusi?n - Random Forest")
plt.show()


In [ ]:

# ? 7. Importancia de las variables
importances = pd.DataFrame({
    "Feature": features,
    "Importance": rf.feature_importances_
}).sort_values(by="Importance", ascending=False)


In [ ]:


sns.barplot(x="Importance", y="Feature", data=importances)
plt.title("Importancia de las variables (Random Forest)")
plt.show()


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report
from scipy.stats import randint

In [ ]:
df.columns

In [ ]:
features

In [ ]:

# Definir el espacio de b?squeda
param_dist = {
    'n_estimators': randint(100,  2000),
    'max_depth': [5, 10, 20, 50, None],
    'min_samples_split': [2, 5, 10,50],
    'min_samples_leaf': [1, 2, 4, 10],
    'max_features': ['sqrt', 'log2', None],
    'class_weight': ['balanced',"balanced_subsample", None]
}


In [ ]:


# Inicializar el modelo base
rf = RandomForestClassifier(random_state=42)

# RandomizedSearchCV
random_search = RandomizedSearchCV(
    rf, param_distributions=param_dist,
    n_iter=50, cv=5, scoring='f1', random_state=42, n_jobs=-1, verbose=1
)

# Ajustar al set de entrenamiento
random_search.fit(X_train, y_train)

# Mejor modelo
best_rf = random_search.best_estimator_

# Predicciones y evaluaci?n
y_pred = best_rf.predict(X_test)
print("Mejores par?metros:", random_search.best_params_)
print(classification_report(y_test, y_pred))


In [ ]:
from sklearn.model_selection import GridSearchCV

# Espacio de b?squeda m?s acotado
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced']
}


In [ ]:

grid_search = GridSearchCV(
    rf, param_grid=param_grid,
    cv=5, scoring='f1', n_jobs=-1, verbose=1
)

grid_search.fit(X_train, y_train)
best_rf_grid = grid_search.best_estimator_

# Predicciones y evaluaci?n
y_pred_grid = best_rf_grid.predict(X_test)
print("Mejores par?metros (Grid):", grid_search.best_params_)
print(classification_report(y_test, y_pred_grid))


In [ ]:
from sklearn.metrics import classification_report

# Supongamos que ya tienes:
# y_test: valores reales
# y_pred: predicciones del modelo
# X_test: contiene columna log10_Value

# Combinar todo en un DataFrame
results = X_test.copy()
results["y_true"] = y_test.values
results["y_pred"] = y_pred

# Agregar una columna con los rangos de log10_Value
def categorize_log10(val):
    if val < 2:
        return "<100"
    elif val < 3:
        return "100?999"
    else:
        return "1000+"

results["Value_range"] = results["log10_Value"].apply(categorize_log10)

# Evaluar por cada rango
for group, subset in results.groupby("Value_range"):
    print(f"?? Rango: {group}")
    print(classification_report(subset["y_true"], subset["y_pred"]))
    print("-" * 40)


In [ ]:
from sklearn.metrics import classification_report

# Suponiendo que ya tienes: X_test, y_test, y_pred
# Aseg?rate de que X_test tenga la columna "log10_Value"

# Combinar datos reales, predichos y caracter?sticas
results = X_test.copy()
results["y_true"] = y_test.values
results["y_pred"] = y_pred

# Filtrar solo casos con log10_Value >= 2 (equivale a LR ? 100)
subset = results[results["log10_Value"] >= 3]

# Reporte de clasificaci?n en ese subconjunto
print("?? Evaluaci?n solo con log10_Value ? 3 (LR ? 1000)")
print(classification_report(subset["y_true"], subset["y_pred"]))


In [ ]:
subset_low = results[results["log10_Value"] < 2]
print("?? Evaluaci?n con log10_Value < 2 (LR < 100)")
print(classification_report(subset_low["y_true"], subset_low["y_pred"]))


In [ ]:
features = ["k1", "k2", "Share_allele", "log10_Value"]
X = df[features]
y = df["True_positive"]  # o el nombre de tu variable objetivo


In [ ]:
pip install xgboost


In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report
from scipy.stats import randint, uniform
import numpy as np

In [ ]:
#features = ["k1", "k2", "Share_allele", "log10_Value"]
features = ["k1", "k2", "variable_rangos_encoded"]

X = df[features]
y = df["True_positive"]  # o el nombre de tu variable objetivo


In [ ]:
# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# Definir el modelo base
xgb_model = XGBClassifier(eval_metric='aucpr')

# Definir espacio de b?squeda para RandomizedSearchCV
param_dist = {
    'n_estimators': randint(100, 2000),  # n?mero de ?rboles
    'max_depth': [5, 10, 20, 50],        # profundidad del ?rbol
    'learning_rate': uniform(0.01, 0.3), # tasa de aprendizaje
    'min_child_weight': [1, 3, 5],       # similar a min_samples_split
    'gamma': [0, 0.1, 0.2],              # reducci?n de impureza m?nima
    'subsample': [0.6, 0.8, 1.0],        # fracci?n de muestras usadas por ?rbol
    'colsample_bytree': [0.6, 0.8, 1.0], # similar a max_features
    'scale_pos_weight': [1, 2, 5],       # para clases desbalanceadas (en vez de class_weight)
}

# Configurar RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=50,  # n?mero de combinaciones aleatorias a probar
    cv=5,       # validaci?n cruzada 5-fold
    scoring='f1',  # puedes usar 'accuracy', 'recall', 'roc_auc', etc.
    verbose=1,
    random_state=42,
    n_jobs=-1
)

In [ ]:

# Ejecutar b?squeda
random_search.fit(X_train, y_train)

# Predecir con el mejor modelo
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)

# Reporte de m?tricas
print("Mejores par?metros encontrados:")
print(random_search.best_params_)
print("\nReporte de clasificaci?n:")
print(classification_report(y_test, y_pred))

In [ ]:
features = ["k0","k1", "k2", "Share_allele","Shared_Markers", "log10_Value"]
X = df[features]
y = df["True_positive"]  # o el nombre de tu variable objetivo

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)

print(classification_report(y_test, y_pred))


In [ ]:
X[["k1", "k2", "Share_allele", "log10_Value"]].corr()


In [ ]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from scipy.stats import uniform, reciprocal
import numpy as np


In [ ]:
X = df[["k1", "k2", "variable_rangos_encoded"]]
y = df["True_positive"]

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)

# Escalado
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
svc = SVC(probability=True)

param_dist = {
    'C': uniform(0.1, 10),  # valores entre 0.1 y 10
    'kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],  # solo para kernels no lineales
    'degree': [2, 3, 4],  # solo para kernel='poly'
}


In [ ]:
random_search = RandomizedSearchCV(
    svc,
    param_distributions=param_dist,
    n_iter=50,  # n?mero de combinaciones a probar
    scoring='f1',  # puedes usar 'recall', 'precision', 'accuracy', etc.
    cv=5,  # validaci?n cruzada con 5 folds
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_scaled, y_train)


In [ ]:
print("Mejores hiperpar?metros encontrados:")
print(random_search.best_params_)

best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test_scaled)

print("\nReporte de clasificaci?n:")
print(classification_report(y_test, y_pred))
